# **Data Ingestion**

In [26]:
import os, glob
import pandas as pd
import re
import numpy as np

In [27]:
from google.colab import drive

drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/DS_project'

# 1) base sample (2024-25)
base = pd.read_parquet(f'{FOLDER}/model_df_m4.parquet')
print('base:', base.shape)

# 2) every .dta in the 2018-19 folder
W18 = f'{FOLDER}/statafiles_2018-19'
paths = sorted(glob.glob(f'{W18}/*.dta'))
print(len(paths), 'files found\n')

raw18 = {}
for p in paths:
    name = os.path.basename(p).replace('.dta', '')
    try:
        df = pd.read_stata(p, convert_categoricals=True)
    except Exception:
        df = pd.read_stata(p, convert_categoricals=False)   # fallback if labels break
        name += ' [no labels]'
    raw18[name] = df

# 3) overview: size + which id columns each file has
keys = ['hhcode', 'prcode', 'hhno', 'idc']
rows = [{'file': n, 'rows': d.shape[0], 'cols': d.shape[1],
         'id_cols': [k for k in keys if k in d.columns],
         'first_cols': list(d.columns[:6])} for n, d in raw18.items()]
pd.set_option('display.width', 250, 'display.max_colwidth', 80)
print(pd.DataFrame(rows).to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
base: (25237, 271)
31 files found

                  file    rows  cols       id_cols                                       first_cols
                 plist  159949    19 [hhcode, idc]     [hhcode, province, region, psu, idc, s1aq02]
                sec 1a  175690    18 [hhcode, idc]     [hhcode, province, region, psu, idc, s1aq02]
                sec 2c  115632    35 [hhcode, idc]     [hhcode, psu, province, region, idc, s2cq01]
                sec_00   25676    22      [hhcode] [hhcode, province, region, psu, ebcode, batchno]
   sec_10a [no labels]  127446    12 [hhcode, idc]  [hhcode, province, region, psu, idc, occu_type]
   sec_10b [no labels]   85051    15 [hhcode, idc]  [hhcode, province, region, psu, idc, occu_type]
  sec_11ab [no labels]  102876    13 [hhcode, idc]  [hhcode, province, region, psu, idc, occu_type]
               sec_12a   63876    13

In [28]:
for p in paths:
    name = os.path.basename(p).replace('.dta', '')
    with pd.read_stata(p, iterator=True) as r:
        labs = r.variable_labels()
    print(f'\n===== {name}  ({len(labs)} cols) =====')
    for c, lab in labs.items():
        print(f'{c:12s} {lab}')


===== plist  (19 cols) =====
hhcode       
province     
region       
psu          
idc          
s1aq02       relationship with the head of household 
s1aq03        reason to become head
s1aq04       gender of person
s1aq05       residensial status
age          age in complete year
s1aq62       days
s1aq63       month
s1aq64       year of birth
s1aq07       marital status
s1aq08       spouse idc
s1aq09       father idc
s1aq10       mother idc
s1aq11       is ---- a  member of the household
weights      

===== sec 1a  (18 cols) =====
hhcode       
province     
region       
psu          
idc          
s1aq02       relationship with the head of household 
s1aq03        reason to become head
s1aq04       gender of person
s1aq05       residensial status
age          age in complete year
s1aq62       days
s1aq63       month
s1aq64       year of birth
s1aq07       marital status
s1aq08       spouse idc
s1aq09       father idc
s1aq10       mother idc
s1aq11       is ---- a  member of the

In [29]:
def read_stata_labeled(path):
    try:
        return pd.read_stata(path, convert_categoricals=True)
    except ValueError as e:
        if "not unique" not in str(e):
            print('  [!] other label error, loading without labels:', e)
            return pd.read_stata(path, convert_categoricals=False)
    with pd.io.stata.StataReader(path) as sr:
        df = sr.read(convert_categoricals=False)
        varlist, lbllist, value_labels = sr._varlist, sr._lbllist, sr.value_labels()
    for col, lblname in zip(varlist, lbllist):
        if lblname and lblname in value_labels and col in df.columns:
            seen, mapping = {}, {}
            for code, text in value_labels[lblname].items():
                seen[text] = seen.get(text, 0) + 1
                mapping[code] = text if seen[text] == 1 else f"{text} ({seen[text]})"
            df[col] = df[col].map(mapping).fillna(df[col])
    return df

L = lambda n: read_stata_labeled(f'{W18}/{n}.dta')
plist, sec1a, edu, emp = L('plist'), L('sec 1a'), L('sec_2ab'), L('sec_1b (2)')
hh, cover, a7 = L('sec_5a'), L('sec_00'), L('sec_7a')

checks = [('plist', plist, ['s1aq04', 's1aq11']), ('sec 1a', sec1a, ['s1aq11']),
          ('sec_2ab', edu, ['s2bq01', 's2bq05', 's2bq14']), ('sec_1b (2)', emp, ['s1bq01']),
          ('sec_5a', hh, ['region', 'province', 's5aq10', 's5aq11', 's5aq21', 's5aq30_1a']),
          ('sec_00', cover, ['interv_status'])]
for name, d, cols in checks:
    for c in cols:
        print(f'\n--- {name}.{c}')
        print(d[c].value_counts(dropna=False).head(25))

print('\nplist unique hhcode+idc:', not plist.duplicated(['hhcode', 'idc']).any())
print('sec 1a unique hhcode+idc:', not sec1a.duplicated(['hhcode', 'idc']).any())
print('sec_5a unique hhcode:', hh.hhcode.is_unique, '| sec_00 unique hhcode:', cover.hhcode.is_unique)
print('plist weights non-missing:', plist['weights'].notna().mean())
print('\nsec_7a codes:', list(a7['code'].dropna().unique())[:60])


--- plist.s1aq04
s1aq04
female    80114
male      79835
Name: count, dtype: int64

--- plist.s1aq11
s1aq11
yes    159949
Name: count, dtype: int64

--- sec 1a.s1aq11
s1aq11
yes    175463
no        227
Name: count, dtype: int64

--- sec_2ab.s2bq01
s2bq01
never attended school                  63318
attended school/institution in past    44349
currently attending school             40047
Name: count, dtype: int64

--- sec_2ab.s2bq05
s2bq05
NaN                                     103365
class10/ o-level                          9626
class5                                    9320
class8                                    6325
fa/f.sc/icom/ics/i.com etc                4765
ba/b.sc/b.com                             1937
ma/msc                                    1741
class4                                    1541
class6                                    1504
class7                                    1471
class3                                    1388
class9/ o-level                         

In [30]:
def ids(d):
    d = d.copy()
    for k in ['hhcode', 'idc']:
        d[k] = pd.to_numeric(d[k], errors='coerce').astype('Int64')
    return d

p, e, w = ids(plist), ids(edu), ids(emp)

r18 = p[['hhcode', 'idc', 'province', 'region', 'age', 's1aq04', 's1aq09', 's1aq10']].rename(columns={
    'age': 's1aq51', 's1aq04': 'sex', 's1aq09': 'father_idc', 's1aq10': 'mother_idc'})
r18 = r18.merge(e[['hhcode', 'idc', 's2bq01', 's2bq05', 's2bq14']], on=['hhcode', 'idc'], how='left', validate='1:1')
r18 = r18.merge(w[['hhcode', 'idc', 's1bq01']], on=['hhcode', 'idc'], how='left', validate='1:1')

for c in ['sex', 'province', 'region', 's2bq01', 's2bq05', 's2bq14', 's1bq01']:
    r18[c] = r18[c].astype('string').str.strip().str.lower()
for c in ['s1aq51', 'father_idc', 'mother_idc']:
    r18[c] = pd.to_numeric(r18[c], errors='coerce')

print('rows:', len(r18), '(plist:', len(plist), ')')
print(r18.dtypes, '\n')
print(r18['s1aq51'].describe(), '\n')
print('mother_idc non-missing:', r18['mother_idc'].notna().sum())
print(r18['mother_idc'].value_counts(dropna=False).head(8), '\n')

girls = r18[(r18.sex == 'female') & r18.s1aq51.between(6, 16)]
print('girls 6-16:', len(girls))
print(girls['s2bq01'].value_counts(dropna=False))

rows: 159949 (plist: 159949 )
hhcode                 Int64
idc                    Int64
province      string[python]
region        string[python]
s1aq51                  int8
sex           string[python]
father_idc              int8
mother_idc              int8
s2bq01        string[python]
s2bq05        string[python]
s2bq14        string[python]
s1bq01        string[python]
dtype: object 

count    159949.000000
mean         23.869327
std          19.027676
min           0.000000
25%           8.000000
50%          19.000000
75%          35.000000
max          99.000000
Name: s1aq51, dtype: float64 

mother_idc non-missing: 159949
mother_idc
51    78263
98    31814
99    29096
52     6655
53     3563
54     2710
55     2260
56     1882
Name: count, dtype: int64 

girls 6-16: 21998
s2bq01
currently attending school             13277
never attended school                   7019
attended school/institution in past     1702
Name: count, dtype: Int64


In [31]:
# 98/99 = no parent in household -> NaN
r18['mother_idc_raw'] = r18['mother_idc']
for c in ['father_idc', 'mother_idc']:
    r18[c] = r18[c].astype(float).where(~r18[c].isin([98, 99]))

# mother table from female records only
cols = ['s1aq51', 's2bq01', 's2bq05', 's2bq14', 's1bq01']
mom = r18.loc[r18.sex == 'female', ['hhcode', 'idc'] + cols].copy()
mom.columns = ['hhcode', 'mother_idc'] + ['mom_' + c for c in ['age', 's2bq01', 's2bq05', 's2bq14', 's1bq01']]
mom['mother_idc'] = mom['mother_idc'].astype(float)

n0 = len(r18)
r18 = r18.merge(mom, on=['hhcode', 'mother_idc'], how='left', validate='m:1')
assert len(r18) == n0, 'merge changed row count'

r18['mother_linked'] = r18['mom_age'].notna().astype(int)
gap = r18['mom_age'] - r18['s1aq51']
r18['mom_age_gap'] = gap
r18['gap_implausible'] = ((gap < 12) | (gap > 55)).where(gap.notna())

g = r18[(r18.sex == 'female') & r18.s1aq51.between(6, 16)]
print('girls 6-16:', len(g))
print('mother linked:', g.mother_linked.sum(), f'({g.mother_linked.mean():.1%})')
print('raw mother code 98 / 99:', (g.mother_idc_raw == 98).sum(), '/', (g.mother_idc_raw == 99).sum())
print('has a mother id but not found as a female:', (g.mother_idc.notna() & (g.mother_linked == 0)).sum())
print('implausible age gap: all rows', int(r18.gap_implausible.sum()), '| girls', int(g.gap_implausible.sum()))

r18 = r18[r18.gap_implausible != True].copy()
print('rows after dropping implausible links:', len(r18))

girls 6-16: 21998
mother linked: 21268 (96.7%)
raw mother code 98 / 99: 457 / 273
has a mother id but not found as a female: 0
implausible age gap: all rows 49 | girls 4
rows after dropping implausible links: 159900


In [32]:
EXTRA_GRADES = {'polytechnic diploma/diploma*': ('higher', np.nan),
                'polytechnic diploma/diploma':  ('higher', np.nan)}   # 2018-19 label has no asterisk
EDU_ORDER = ['none', 'primary', 'middle', 'secondary', 'higher']

def map_grade(lab):
    if not isinstance(lab, str):
        return (np.nan, np.nan)
    t = lab.strip().lower()
    if t in ('', 'not applicable', 'nan'):
        return (np.nan, np.nan)
    if t in EXTRA_GRADES:
        return EXTRA_GRADES[t]
    m = re.search(r'class\s*(\d+)', t)
    if m:
        n = int(m.group(1))
        lev = 'primary' if n <= 5 else 'middle' if n <= 8 else 'secondary' if n <= 10 else 'higher'
        return (lev, n)
    if re.search(r'katchi|nursery|prep|kindergarten|\bkg\b|pre-?primary|play ?group', t):
        return ('primary', 0)
    if 'matric' in t and 'below' not in t:
        return ('secondary', 10)
    if re.search(r'm\.?\s?phil|ph\.?\s?d', t):
        return ('higher', 18)
    if re.search(r'\bdegree\b', t):
        return ('higher', 16)
    if re.search(r'^ms$|\bms\b', t):
        return ('higher', 18)
    if re.search(r'bachelor|\bb\.?a\b|\bb\.?sc|\bb\.?com|\bb\.?ed\b|\bbba\b|\bbs\b|\bmbbs\b|\bbe\b', t):
        return ('higher', 16 if ('4' in t and 'year' in t) else 14)
    if re.search(r'\bm\.?a\b|\bm\.?sc\b|master|\bmba\b|\bm\.?com\b', t):
        return ('higher', 16)
    if re.search(r'\bf\.?a\b|f\.?sc|i\.?com|\bics\b|a-?level|intermediate|hssc', t):
        return ('higher', 12)
    return None

def educ(df, cur_col):
    st = df['mom_s2bq01'].astype(object)
    lev = pd.Series(np.nan, index=df.index, dtype=object)
    yrs = pd.Series(np.nan, index=df.index, dtype=float)
    never = st.astype(str).str.contains('never') & st.notna()
    past  = st.astype(str).str.contains('past') & st.notna()
    cur   = st.astype(str).str.contains('current') & st.notna()
    lev[never], yrs[never] = 'none', 0.0
    unmapped = {}
    for mask, col in [(past, 'mom_s2bq05'), (cur, cur_col)]:
        lab = df.loc[mask, col].astype(object)
        mp = {u: map_grade(u) for u in lab.dropna().unique()}
        for u, v in mp.items():
            if v is None:
                unmapped[u] = int((lab == u).sum())
        lev[mask] = lab.map(lambda v: mp[v][0] if (v in mp and mp[v] is not None) else np.nan)
        yrs[mask] = lab.map(lambda v: mp[v][1] if (v in mp and mp[v] is not None) else np.nan).astype(float)
    print('unmapped mother grade labels (-> NaN):', unmapped)
    return pd.Categorical(lev, categories=EDU_ORDER, ordered=True), yrs

g18 = r18[(r18.sex == 'female') & r18.s1aq51.between(6, 16)].copy()

st = g18['s2bq01']
g18['out_of_school'] = np.where(st.str.contains('current'), 0.0,
                        np.where(st.str.contains('never|past'), 1.0, np.nan))
g18['rural'] = (g18['region'] == 'rural').astype(int)
g18['province'] = g18['province'].replace({'khyber paktunkhwa': 'kp'})
g18['hh_size'] = g18['hhcode'].map(p.groupby('hhcode').size())
g18['mom_educ_level'], g18['mom_edu_years'] = educ(g18, cur_col='mom_s2bq14')
g18['mom_employed'] = g18['mom_s1bq01'].astype(object).map({'yes': 1, 'no': 0}).astype(float)

print('\ngirls:', len(g18), '| households:', g18.hhcode.nunique())
print('out_of_school:', g18.out_of_school.value_counts(dropna=False).to_dict())
print('province:', g18.province.value_counts().to_dict())
print('rural share:', round(g18.rural.mean(), 3))
print('\nmom_educ_level %:\n', g18.mom_educ_level.value_counts(normalize=True, dropna=False).round(3))
print('\nmom_employed %:\n', g18.mom_employed.value_counts(normalize=True, dropna=False).round(3))
print('\nmissing: mom_age', g18.mom_age.isna().sum(), '| mom_educ_level', g18.mom_educ_level.isna().sum(),
      '| mom_employed', g18.mom_employed.isna().sum())

unmapped mother grade labels (-> NaN): {'other': 4}

girls: 21994 | households: 12748
out_of_school: {0.0: 13274, 1.0: 8720}
province: {'punjab': 9056, 'sindh': 5247, 'khyber pakhtunkhwa': 4969, 'balochistan': 2722}
rural share: 0.672

mom_educ_level %:
 mom_educ_level
none         0.698
primary      0.100
secondary    0.062
higher       0.055
middle       0.051
NaN          0.034
Name: proportion, dtype: float64

mom_employed %:
 mom_employed
0.0    0.702
1.0    0.265
NaN    0.033
Name: proportion, dtype: float64

missing: mom_age 730 | mom_educ_level 740 | mom_employed 730


In [33]:
def norm(s):
    return s.astype('string').str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def group_label(series, rules, default='other/unmapped'):
    s = norm(series)
    out = pd.Series(default, index=series.index, dtype='object')
    done = pd.Series(False, index=series.index)
    for label, pats in rules:
        hit = s.str.contains('|'.join(pats), regex=True, na=False) & ~done
        out[hit] = label
        done |= hit
    out[series.isna()] = np.nan
    return out

WATER_RULES = [
    ('surface',    [r'river', r'canal', r'stream', r'pond', r'lake', r'surface']),
    ('unimproved', [r'open well', r'un ?protected', r'tanker', r'vendor', r'cart', r'^other$']),
    ('improved',   [r'tap', r'pipe', r'pump', r'tube ?well', r'bore', r'closed well',
                    r'protected', r'spring', r'bottle', r'mineral', r'filtration']),
]
TOILET_RULES = [
    ('none',       [r'no toilet', r'^none$', r'open (field|defecation)', r'bush']),
    ('improved',   [r'flush', r'septic', r'sewer']),
    ('unimproved', [r'latrine', r'dry', r'pit', r'bucket', r'hanging', r'^other$']),
]

h = hh.copy()
h['hhcode'] = pd.to_numeric(h['hhcode'], errors='coerce').astype('Int64')
h['has_electricity'] = np.where(h['s5aq10'].isna(), np.nan,
                                norm(h['s5aq10']).str.contains('electricity').fillna(False).astype(float))
h['water_group']  = group_label(h['s5aq11'], WATER_RULES)
h['toilet_group'] = group_label(h['s5aq21'], TOILET_RULES)

# CHECK these two tables: every raw label should land in the group you expect
display(pd.crosstab(norm(h['s5aq11']), h['water_group']))
display(pd.crosstab(norm(h['s5aq21']), h['toilet_group']))

# province fix + merge household variables and weight
n0 = len(g18)
g18['province'] = g18['province'].replace({'khyber pakhtunkhwa': 'kp'})
g18 = g18.merge(h[['hhcode', 'has_electricity', 'water_group', 'toilet_group']],
                on='hhcode', how='left', validate='m:1')
g18 = g18.merge(p[['hhcode', 'idc', 'weights']].rename(columns={'weights': 'weight'}),
                on=['hhcode', 'idc'], how='left', validate='1:1')
assert len(g18) == n0, 'merge changed row count'

print('\nprovince:', g18.province.value_counts().to_dict())
print('missing:', g18[['has_electricity', 'water_group', 'toilet_group', 'weight']].isna().sum().to_dict())
print('has_electricity mean:', round(g18.has_electricity.mean(), 3))
print(g18.water_group.value_counts(normalize=True).round(3).to_dict())
print(g18.toilet_group.value_counts(normalize=True).round(3).to_dict())
print(g18.weight.describe().round(1))

/tmp/ipykernel_820/1903168806.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = s.str.contains('|'.join(pats), regex=True, na=False) & ~done


water_group,improved,other/unmapped,surface,unimproved
s5aq11,,,,
"bottled water,",510,0,0,0
closed well,341,0,0,0
closed well (2),180,0,0,0
filtration plant,1811,0,0,0
hand pump,4609,0,0,0
hand pump (2),1523,0,0,0
motor pump /tube well,1457,0,0,0
"motor pump /tube well,",6728,0,0,0
open well,0,0,0,363


toilet_group,improved,none,other/unmapped,unimproved
s5aq21,,,,
composting toilet,0,0,25,0
dry pit latrine,0,0,0,1674
dry raised latrine,0,0,0,991
flush connected to open drain,3337,0,0,0
flush connected to pit,4536,0,0,0
flush connected to public sewerage,5412,0,0,0
flush connected to septic tank,5674,0,0,0
no toilet,0,3113,0,0
other (specify_____________),0,0,47,0



province: {'punjab': 9056, 'sindh': 5247, 'kp': 4969, 'balochistan': 2722}
missing: {'has_electricity': 0, 'water_group': 0, 'toilet_group': 0, 'weight': 0}
has_electricity mean: 0.878
{'improved': 0.882, 'unimproved': 0.087, 'surface': 0.028, 'other/unmapped': 0.003}
{'improved': 0.736, 'unimproved': 0.138, 'none': 0.123, 'other/unmapped': 0.003}
count    21994.0
mean      1280.6
std        500.6
min        367.8
25%        899.4
50%       1332.0
75%       1591.6
max       3454.7
Name: weight, dtype: float64


In [34]:
# 1) phone/computer assets in sec_5a
for c in ['s5aq30_1a', 's5aq30_2a', 's5aq30_3a', 's5aq30_4a', 's5aq30_5a', 's5aq30_6a']:
    print(c, norm(hh[c]).value_counts(dropna=False).to_dict())

# 2) radio and TV in sec_7a
a = a7.copy()
a['hhcode'] = pd.to_numeric(a['hhcode'], errors='coerce').astype('Int64')
a['item'] = norm(a['code'])
sel = a[a['item'].isin(['radio', 'televsion', 'lcd/ led'])]
print('\nrows per item:', sel['item'].value_counts().to_dict())
print('unique hhcode+item:', not sel.duplicated(['hhcode', 'item']).any())
print('households in sec_7a:', a['hhcode'].nunique(), '| in sec_5a:', hh['hhcode'].nunique())
print('rows per household (all items):', a.groupby('hhcode').size().describe().round(1).to_dict())
print(sel.head(6))
for it in ['radio', 'televsion', 'lcd/ led']:
    s = sel[sel['item'] == it]
    print(f'\n--- {it}')
    print('c01:', s['c01'].value_counts(dropna=False).head(8).to_dict())
    print('c02:', s['c02'].value_counts(dropna=False).head(8).to_dict())

# 3) are the 2024-25 asset flags still in the base parquet?
print('\nbase _bin columns:', [c for c in base.columns if c.endswith('_bin')])
print('base has wealth_index:', 'wealth_index' in base.columns)

s5aq30_1a {'no': 16769, 'yes': 8040}
s5aq30_2a {'yes': 23356, 'no': 1453}
s5aq30_3a {'no': 24338, 'yes': 471}
s5aq30_4a {'no': 23056, 'yes': 1753}
s5aq30_5a {'no': 23271, 'yes': 1538}
s5aq30_6a {'no': 24415, 'yes': 394}

rows per item: {'televsion': 12252, 'lcd/ led': 2664, 'radio': 949}
unique hhcode+item: True
households in sec_7a: 24114 | in sec_5a: 24809
rows per household (all items): {'count': 24114.0, 'mean': 8.1, 'std': 4.4, 'min': 1.0, '25%': 5.0, '50%': 8.0, '75%': 11.0, 'max': 25.0}
        hhcode            province region       psu       code  c01  c02  c03       item
9   1102100101  khyber pakhtunkhwa  urban  11021001  televsion  1.0  1.0  NaN  televsion
29  1102100103  khyber pakhtunkhwa  urban  11021001   lcd/ led  1.0  1.0  NaN   lcd/ led
40  1102100105  khyber pakhtunkhwa  urban  11021001  televsion  1.0  1.0  NaN  televsion
47  1102100106  khyber pakhtunkhwa  urban  11021001  televsion  1.0  1.0  NaN  televsion
53  1102100107  khyber pakhtunkhwa  urban  11021001  tel

In [35]:
# ===== STEP 5: WEALTH INDEX on the 8 assets both waves share =====
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

ASSETS = ['internet', 'mobile', 'landline', 'computer', 'laptop', 'tablet', 'radio', 'tv']
Q_LABELS = ['Q1 - Lowest', 'Q2', 'Q3', 'Q4', 'Q5 - Highest']
yn = lambda s: (norm(s) == 'yes').astype(float)

# 2018-19 household flags (sec_5a) + radio/TV (sec_7a)
a18 = pd.DataFrame({'hhcode': h['hhcode'],
                    'internet': yn(h['s5aq30_1a']), 'mobile': yn(h['s5aq30_2a']),
                    'landline': yn(h['s5aq30_3a']), 'computer': yn(h['s5aq30_4a']),
                    'laptop': yn(h['s5aq30_5a']), 'tablet': yn(h['s5aq30_6a'])})
d7 = a7.copy()
d7['hhcode'] = pd.to_numeric(d7['hhcode'], errors='coerce').astype('Int64')
d7['item'] = norm(d7['code'])
owned = pd.to_numeric(d7['c02'], errors='coerce').fillna(pd.to_numeric(d7['c01'], errors='coerce')) > 0
radio_hh = d7.loc[owned & (d7['item'] == 'radio'), 'hhcode'].dropna().unique().tolist()
tv_hh = d7.loc[owned & d7['item'].isin(['televsion', 'lcd/ led']), 'hhcode'].dropna().unique().tolist()
a18['radio'] = a18['hhcode'].isin(radio_hh).astype(float)     # household absent from sec_7a = owns none of the listed items
a18['tv'] = a18['hhcode'].isin(tv_hh).astype(float)
print('2018-19 households:', len(a18), '| not in sec_7a (radio=tv=0 assumed):', int((~a18.hhcode.isin(d7.hhcode.unique())).sum()))

# same 8 flags for the 2024-25 rows (mobile = simple OR smart phone)
def base_flags(df):
    return pd.DataFrame({'internet': df['s5m1q111a_bin'],
                         'mobile': df[['s5m1q112a_bin', 's5m1q113a_bin']].max(axis=1),
                         'landline': df['s5m1q114a_bin'], 'computer': df['s5m1q115a_bin'],
                         'laptop': df['s5m1q116a_bin'], 'tablet': df['s5m1q117a_bin'],
                         'radio': df['s5m1q118a_bin'], 'tv': df['s5m1q119a_bin']}).astype(float)

bh = base.drop_duplicates(['prcode', 'hhno'])
Xb_hh = base_flags(bh)
assert not Xb_hh.isna().any().any(), 'NaN in base asset flags'

# fit once on 2024-25 households, apply unchanged to both waves
sc = StandardScaler().fit(Xb_hh[ASSETS])
pca = PCA(n_components=1).fit(sc.transform(Xb_hh[ASSETS]))
raw_hh = pca.transform(sc.transform(Xb_hh[ASSETS])).ravel()
sign = 1 if np.corrcoef(raw_hh, Xb_hh[ASSETS].sum(axis=1))[0, 1] > 0 else -1
score = lambda X: sign * pca.transform(sc.transform(X[ASSETS])).ravel()
edges = np.quantile(sign * raw_hh, [0.2, 0.4, 0.6, 0.8])             # quintile cut-points from 2024-25 households
quint = lambda idx: pd.Series(np.array(Q_LABELS, dtype=object)[np.searchsorted(edges, idx, side='left')])

print('PC1 variance explained:', round(pca.explained_variance_ratio_[0], 3))
base2 = base.copy()
base2['wealth_index_orig'] = base2['wealth_index']
base2['wealth_index'] = score(base_flags(base2))
base2['wealth_quintile'] = quint(base2['wealth_index'].values).values
print('corr(new shared index, original 9-asset index) in 2024-25:',
      round(base2[['wealth_index', 'wealth_index_orig']].corr().iloc[0, 1], 3))

a18['wealth_index'] = score(a18)
a18['wealth_quintile'] = quint(a18['wealth_index'].values).values
n0 = len(g18)
g18 = g18.drop(columns=[c for c in ['wealth_index', 'wealth_quintile'] if c in g18.columns])
g18 = g18.merge(a18[['hhcode', 'wealth_index', 'wealth_quintile']], on='hhcode', how='left', validate='m:1')
assert len(g18) == n0
print('\nwealth_index mean  2024-25:', round(base2.wealth_index.mean(), 3), '| 2018-19:', round(g18.wealth_index.mean(), 3))
print('quintile shares 2024-25:', base2.wealth_quintile.value_counts(normalize=True).round(3).sort_index().to_dict())
print('quintile shares 2018-19:', g18.wealth_quintile.value_counts(normalize=True).round(3).sort_index().to_dict())
print('missing wealth_index in 2018-19 girls:', g18.wealth_index.isna().sum())

2018-19 households: 24809 | not in sec_7a (radio=tv=0 assumed): 695
PC1 variance explained: 0.198
corr(new shared index, original 9-asset index) in 2024-25: 0.811

wealth_index mean  2024-25: -0.028 | 2018-19: -0.074
quintile shares 2024-25: {'Q1 - Lowest': 0.295, 'Q2': 0.458, 'Q4': 0.155, 'Q5 - Highest': 0.092}
quintile shares 2018-19: {'Q1 - Lowest': 0.366, 'Q2': 0.382, 'Q4': 0.135, 'Q5 - Highest': 0.118}
missing wealth_index in 2018-19 girls: 0


In [36]:
# ===== STEP 6: IDs + one schema (same columns, dtypes, categories) for both waves =====
BASE_WAVE, NEW_WAVE = 'HIES 2024-25', 'HIES 2018-19'

# 2018-19 ids in the 2024-25 style: prcode = psu, hhno = last digits of hhcode
psu = pd.to_numeric(p.drop_duplicates('hhcode').set_index('hhcode')['psu'], errors='coerce')
g18['prcode'] = g18['hhcode'].map(psu).astype('Int64')
g18['hhno'] = (g18['hhcode'] - g18['prcode'] * 100).astype('Int64')
assert g18['hhno'].between(1, 99).all(), 'hhcode is not psu*100 + hhno'
# household size counted after dropping implausible links, as in 2024-25
g18['hh_size'] = g18['hhcode'].map(r18.groupby('hhcode').size())

LEVELS = {'province': ['punjab', 'sindh', 'kp', 'balochistan'],
          'water_group': ['improved', 'unimproved', 'surface', 'other/unmapped'],
          'toilet_group': ['improved', 'unimproved', 'none', 'other/unmapped'],
          'mom_educ_level': EDU_ORDER,
          'wealth_quintile': Q_LABELS}
FALLBACK = {'water_group': 'other/unmapped', 'toilet_group': 'other/unmapped'}   # everything else -> NaN
NUM = ['s1aq51', 'hh_size', 'wealth_index', 'wealth_index_orig', 'mom_age', 'mom_age_gap', 'mom_edu_years', 'weight']
BIN = ['out_of_school', 'rural', 'mom_employed', 'has_electricity', 'mother_linked']
CAT = list(LEVELS)
IDS = ['wave', 'hh_uid', 'prcode', 'hhno', 'idc', 'region']
COLS = IDS + NUM + BIN + CAT
SCHEMA_LOG = []

def to_schema(df, wave):
    d = df.copy()
    if 'wealth_index_orig' not in d.columns:
        d['wealth_index_orig'] = np.nan                      # only exists for 2024-25 (9-asset index)
    d['mother_linked'] = d['mom_age'].notna().astype(float)  # same definition in both waves
    for c in NUM + BIN:
        d[c] = pd.to_numeric(d[c], errors='coerce').astype('float64')
    for c, lv in LEVELS.items():
        t = d[c].astype('object').map(lambda v: v.strip() if isinstance(v, str) else v)
        if c != 'wealth_quintile':
            t = t.map(lambda v: v.lower() if isinstance(v, str) else v)
        bad = sorted(set(t.dropna()) - set(lv))
        if bad:
            msg = f'[{wave}] {c}: unmapped levels {bad} -> {FALLBACK.get(c, np.nan)!r}'
            SCHEMA_LOG.append(msg); print(msg)
            t = t.where(~t.isin(bad), FALLBACK.get(c, np.nan))
        d[c] = pd.Categorical(t, categories=lv, ordered=c in ('mom_educ_level', 'wealth_quintile'))
    for c in ['prcode', 'hhno', 'idc']:
        d[c] = pd.to_numeric(d[c], errors='coerce').astype('Int64')
    d['region'] = d['region'].astype('string').str.strip().str.lower()
    d['wave'] = wave
    d['hh_uid'] = wave + '|' + d['prcode'].astype('string') + '|' + d['hhno'].astype('string')
    d = d.dropna(subset=['out_of_school'])
    return d[COLS].reset_index(drop=True)

base_f = to_schema(base2, BASE_WAVE)
new_f = to_schema(g18, NEW_WAVE)
print('\nbase_f:', base_f.shape, '| new_f:', new_f.shape)
print('identical columns:', list(base_f.columns) == list(new_f.columns))
print('identical dtypes :', (base_f.dtypes == new_f.dtypes).all())
print(pd.DataFrame({'base': base_f.dtypes.astype(str), 'new': new_f.dtypes.astype(str)}).T.to_string())


base_f: (25237, 24) | new_f: (21994, 24)
identical columns: True
identical dtypes : True
        wave  hh_uid prcode   hhno    idc  region   s1aq51  hh_size wealth_index wealth_index_orig  mom_age mom_age_gap mom_edu_years   weight out_of_school    rural mom_employed has_electricity mother_linked  province water_group toilet_group mom_educ_level wealth_quintile
base  object  string  Int64  Int64  Int64  string  float64  float64      float64           float64  float64     float64       float64  float64       float64  float64      float64         float64       float64  category    category     category       category        category
new   object  string  Int64  Int64  Int64  string  float64  float64      float64           float64  float64     float64       float64  float64       float64  float64      float64         float64       float64  category    category     category       category        category


In [37]:
# ===== STEP 7: QUALITY AUDIT (missingness + distribution drift, base vs new) =====
rows = []
for c in NUM + BIN + CAT:
    b, n = base_f[c], new_f[c]
    r = {'column': c, 'miss_%_base': round(100 * b.isna().mean(), 2), 'miss_%_new': round(100 * n.isna().mean(), 2)}
    r['gap_pp'] = round(r['miss_%_new'] - r['miss_%_base'], 2)
    if c in NUM + BIN:
        r['mean_base'], r['mean_new'] = round(b.mean(), 3), round(n.mean(), 3)
    r['flag'] = ('ALL MISSING in new' if n.isna().all() else
                 'ALL MISSING in base' if b.isna().all() else
                 'missingness gap > 5pp' if abs(r['gap_pp']) > 5 else '')
    rows.append(r)
audit = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(audit.to_string(index=False))

print('\n--- category shares by wave (rows add to 1) ---')
both = pd.concat([base_f, new_f], ignore_index=True)
for c in CAT:
    print(f'\n{c}')
    print(pd.crosstab(both['wave'], both[c].astype('object').fillna('(missing)'), normalize='index').round(3).to_string())

# hard checks
assert new_f['out_of_school'].notna().all() and base_f['out_of_school'].notna().all()
assert base_f['s1aq51'].between(6, 16).all() and new_f['s1aq51'].between(6, 16).all()
assert not new_f.duplicated(['hh_uid', 'idc']).any() and not base_f.duplicated(['hh_uid', 'idc']).any()
print('\nchecks passed: target present, ages 6-16, no duplicate persons in either wave')

           column  miss_%_base  miss_%_new  gap_pp  mean_base  mean_new               flag
           s1aq51         0.00        0.00    0.00     10.712    10.734                   
          hh_size         0.00        0.00    0.00      8.148     8.485                   
     wealth_index         0.00        0.00    0.00     -0.028    -0.074                   
wealth_index_orig         0.00      100.00  100.00     -0.095       NaN ALL MISSING in new
          mom_age         3.61        3.32   -0.29     39.504    39.133                   
      mom_age_gap         3.61        3.32   -0.29     28.837    28.442                   
    mom_edu_years         3.93        3.37   -0.56      2.994     2.277                   
           weight         0.00        0.00    0.00   1272.062  1280.567                   
    out_of_school         0.00        0.00    0.00      0.345     0.396                   
            rural         0.00        0.00    0.00      0.665     0.672                   

In [38]:
# ===== STEP 8: COMBINE, VERIFY Ntotal / Ptotal, SAVE =====
import json
final = pd.concat([base_f, new_f], ignore_index=True)
for c, lv in LEVELS.items():                                  # keep categories identical after concat
    final[c] = pd.Categorical(final[c].astype('object'), categories=lv, ordered=c in ('mom_educ_level', 'wealth_quintile'))

N_initial, N_new, N_total = len(base_f), len(new_f), len(final)
assert N_total == N_initial + N_new, 'Ntotal != Ninitial + Nnew'
assert not final.duplicated(['hh_uid', 'idc']).any(), 'duplicate person after stacking'
assert final['wave'].value_counts().to_dict() == {BASE_WAVE: N_initial, NEW_WAVE: N_new}

print(f'Ninitial = {N_initial:,} | Nnew = {N_new:,} | Ntotal = {N_total:,}')
print(f'Ptotal = {final.shape[1]} columns  (ids/wave {len(IDS)}, numeric {len(NUM)}, binary {len(BIN)}, categorical {len(CAT)})')
print('households:', final.groupby('wave')['hh_uid'].nunique().to_dict())
print('out_of_school rate by wave:', final.groupby('wave')['out_of_school'].mean().round(3).to_dict())

OUT = f'{FOLDER}/model_df_extended.parquet'
final.to_parquet(OUT, index=False)
new_f.to_parquet(f'{FOLDER}/model_df_2018_19.parquet', index=False)
audit.to_csv(f'{FOLDER}/extension_column_audit.csv', index=False)
json.dump({'N_initial': N_initial, 'N_new': N_new, 'N_total': N_total, 'P_total': int(final.shape[1]),
           'schema_log': SCHEMA_LOG}, open(f'{FOLDER}/extension_audit.json', 'w'), indent=2)
chk = pd.read_parquet(OUT)
print('\nreloaded:', chk.shape, '| categorical dtypes kept:', all(str(chk[c].dtype) == 'category' for c in CAT))

Ninitial = 25,237 | Nnew = 21,994 | Ntotal = 47,231
Ptotal = 24 columns  (ids/wave 6, numeric 8, binary 5, categorical 5)
households: {'HIES 2018-19': 12748, 'HIES 2024-25': 14841}
out_of_school rate by wave: {'HIES 2018-19': 0.396, 'HIES 2024-25': 0.345}

reloaded: (47231, 24) | categorical dtypes kept: True


# Data Extension: adding the HIES 2018-19 survey

## Goal
Add a second survey wave (HIES 2018-19) to our 2024-25 sample of girls aged 6-16, so we can test whether our findings hold in another year and get more data. The new data has to match the format our model already uses: same variables, same categories, same missing-value handling.

## What we did

**1. Loaded the old data.** We read `model_df_m4.parquet`, a copy of the Milestone 4 data saved with the extra infrastructure variables (electricity, water, toilet). This is our starting sample: 25,237 girls.

**2. Loaded all 31 Stata files from the 2018-19 survey** to see what each one contains. Every file was printed with its column names and labels.

**3. Picked the files we need.** Only a few files hold our variables:
- `plist`: age, sex, household members, mother and father links, survey weight
- `sec_2ab`: school attendance and education
- `sec_1b (2)`: employment (used for the mother)
- `sec_5a`: electricity, water, toilet, phone, internet and computer ownership
- `sec_7a`: radio and TV ownership
The other files (health, fertility, consumption, income, land) were not used.

**4. Found what is different in 2018-19.** The variable names mean different things, so we could not just reuse the old code:
- Mother's ID is `s1aq10` (it was `s1aq09` in 2024-25). The 2018-19 `s1aq09` is the father.
- Current school grade is `s2bq14` (it was `s2bq10`).
- Households are identified by `hhcode` only, with no `prcode` or `hhno`.
- Parent codes 98 and 99 mean "no parent in the household", so they become missing.
- Label spellings differ (province names, "polytechnic diploma", "un protected spring", double spaces).
- There is no separate smartphone question, and radio and TV sit in another file.

**5. Rebuilt every variable in the 2024-25 format:**
- Step 1: person table from the roster, education and employment files.
- Step 2: linked each girl to her mother (96.7% linked, similar to 96.4% before) and dropped the few implausible mother-child age gaps.
- Step 3: built the outcome (out of school: 1 = no, 0 = currently attending), rural, province, household size, mother's education, age and employment. We reused the same education-grade rules as before.
- Step 4: added electricity, water group and toilet group with the same rules as Milestone 4, plus the survey weight.
- Step 5: rebuilt the wealth index.
- Step 6: gave both waves the same columns, data types and category levels.
- Step 7: audited missing values and differences between waves.
- Step 8: stacked the two waves, checked Ntotal = Ninitial + Nnew, and saved the result.

## Decisions to remember
- **Wealth index:** 2018-19 has no smartphone question, so the index uses the 8 assets both surveys share (internet, mobile, landline, computer, laptop, tablet, radio, TV). It is fitted on the 2024-25 households and applied unchanged to both waves, so both are on the same scale. The old 9-asset index is kept as `wealth_index_orig` for 2024-25 only.
- **Missing values:** handled the same way as in the earlier milestones. Nothing is imputed here, and unlinked mothers stay missing with `mother_linked` as a flag. No rows are dropped for missing predictors, and imputation happens later inside the model pipeline. Households that don't appear in the radio/TV file are assumed to own neither.
- **Distance to school** is excluded, because it is only recorded for girls currently in school and would leak the answer.
- **Household IDs** can repeat across waves, so each household gets a unique ID that includes the wave name (`hh_uid`). Cross-validation should group by this.
- **Weights** differ by wave and should not be pooled without rescaling.

## Output
`model_df_extended.parquet` (both waves stacked), `model_df_2018_19.parquet` (new wave only), plus a column audit CSV and a JSON file with the final counts.